# 🔍 Is Your Model Fair? A Beginner's Guide to Bias Testing

---

**What you'll learn in this notebook:**
1. What bias in AI models means (and why it matters)
2. How to load a real dataset from HuggingFace 🤗
3. How to test for **sentiment bias** across demographic groups
4. How to test for **toxicity detection fairness**
5. How to test for **accuracy gaps** across groups
6. How to visualize and interpret your results

> 💡 **No ML background needed!** We'll explain every concept as we go.

---

## 🤔 What is Bias in AI?

Imagine you train a model to detect whether a movie review is positive or negative.
Now imagine the model works great for reviews written by men, but often gets it wrong for reviews written by women.

That's **bias** — when a model performs differently (and unfairly) for different groups of people.

**Why does this happen?**
- The training data had more examples from some groups than others
- The training data itself reflected human prejudices
- The model learned patterns that correlate with group identity

**Why does it matter?**
When AI is used for hiring, loans, healthcare, or content moderation — biased models can cause real harm.

## 📦 Step 1: Install and Import Libraries

We'll use:
- `datasets` — to load datasets from HuggingFace
- `transformers` — to load pre-trained models
- `pandas` — to work with data tables
- `matplotlib` / `seaborn` — to make charts

In [ ]:
# Install required packages (run this once)
!pip install datasets transformers torch pandas matplotlib seaborn scikit-learn -q

In [ ]:
# Import everything we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print("✅ All libraries imported successfully!")

---
## 🗂️ Step 2: Load a Real Dataset from HuggingFace

We'll use the **WinoBias** dataset — a benchmark specifically designed to test gender bias in NLP models.

We'll also use the **Civil Comments** dataset (a subset), which tests toxicity detection across demographic groups.

> 🧠 **What is HuggingFace?** It's like a GitHub for AI models and datasets. Thousands of research datasets are freely available there.

In [ ]:
# -------------------------------------------------------
# PART A: Load a sentiment dataset with demographic info
# We'll use the 'md_gender_bias' dataset from HuggingFace
# This dataset contains text labeled with gender information
# -------------------------------------------------------

print("Loading dataset... (this may take a moment)")
dataset = load_dataset("md_gender_bias", "funpedia", trust_remote_code=True)

# Convert to a pandas DataFrame for easier analysis
df = pd.DataFrame(dataset['train'])

print(f"\n✅ Dataset loaded!")
print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn names: {list(df.columns)}")
df.head()

In [ ]:
# Let's understand the gender distribution in the dataset
# gender: 0 = neutral, 1 = masculine, 2 = feminine

gender_map = {0: 'Neutral', 1: 'Masculine', 2: 'Feminine'}
df['gender_label'] = df['gender'].map(gender_map)

gender_counts = df['gender_label'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(gender_counts.index, gender_counts.values, 
               color=['#7CBFCC', '#E8A87C', '#9ED9A6'], edgecolor='white', linewidth=1.5)
ax.set_title('📊 Gender Distribution in the Dataset', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Gender Category', fontsize=12)
ax.set_ylabel('Number of Samples', fontsize=12)
for bar, val in zip(bars, gender_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
            f'{val:,}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n⚠️  Notice anything? If one group has FAR fewer samples, the model may not learn it as well.")
print("    This is called REPRESENTATION BIAS in training data.")

---
## 🎭 Part 1: Sentiment Bias Across Demographic Groups

**The experiment:** We'll run a sentiment analysis model on text associated with different demographic groups and see if it assigns systematically different sentiments.

**The hypothesis:** A fair model should not rate text about one gender as more positive/negative than another, *all else being equal*.

> 🧪 We're using a small sample to keep things fast for the demo. In real research, you'd use thousands of examples.

In [ ]:
# Load a sentiment analysis model from HuggingFace
# This is a pre-trained model — we're NOT training it ourselves,
# just asking it to label text as POSITIVE or NEGATIVE

print("Loading sentiment analysis model...")
sentiment_model = pipeline(
    "sentiment-analysis", 
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)
print("✅ Sentiment model loaded!")

# Let's test it quickly
test = sentiment_model(["I love this!", "This is terrible."])
print(f"\nQuick test:")
print(f"  'I love this!'      → {test[0]['label']} (confidence: {test[0]['score']:.2%})")
print(f"  'This is terrible.' → {test[1]['label']} (confidence: {test[1]['score']:.2%})")

In [ ]:
# Sample equal numbers from each gender group (for a fair comparison!)
# This step itself is important: we MUST compare equal-sized samples

SAMPLE_SIZE = 100  # Keep small for demo speed; increase for real analysis

sample_masc = df[df['gender_label'] == 'Masculine'].sample(SAMPLE_SIZE, random_state=42)
sample_fem  = df[df['gender_label'] == 'Feminine'].sample(SAMPLE_SIZE, random_state=42)
sample_neut = df[df['gender_label'] == 'Neutral'].sample(SAMPLE_SIZE, random_state=42)

print(f"Sampled {SAMPLE_SIZE} texts from each group.")
print(f"\nExample masculine text: {sample_masc['text'].iloc[0][:120]}...")
print(f"\nExample feminine text:  {sample_fem['text'].iloc[0][:120]}...")

In [ ]:
# Run the sentiment model on all samples
# This is the 'bias audit' step

def run_sentiment(texts):
    """Run sentiment analysis and return a DataFrame of results."""
    results = sentiment_model(texts.tolist(), batch_size=16)
    return pd.DataFrame([
        {'label': r['label'], 'score': r['score']} for r in results
    ])

print("Running sentiment model on all groups (may take ~30 seconds)...")

masc_results = run_sentiment(sample_masc['text'])
masc_results['group'] = 'Masculine'

fem_results = run_sentiment(sample_fem['text'])
fem_results['group'] = 'Feminine'

neut_results = run_sentiment(sample_neut['text'])
neut_results['group'] = 'Neutral'

all_results = pd.concat([masc_results, fem_results, neut_results], ignore_index=True)
print("✅ Done!")
all_results.head()

In [ ]:
# Visualize: What % of text in each group is labeled POSITIVE?

positive_rates = all_results.groupby('group').apply(
    lambda x: (x['label'] == 'POSITIVE').mean() * 100
).reset_index()
positive_rates.columns = ['Group', 'Positive Rate (%)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Positive rate by group
colors = ['#7CBFCC', '#E8A87C', '#9ED9A6']
bars = axes[0].bar(positive_rates['Group'], positive_rates['Positive Rate (%)'],
                    color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('% of Text Labeled POSITIVE\nby Gender Group', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Positive Rate (%)')
axes[0].set_ylim(0, 100)
axes[0].axhline(50, color='red', linestyle='--', alpha=0.5, label='50% baseline')
axes[0].legend()
for bar, val in zip(bars, positive_rates['Positive Rate (%)']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Plot 2: Distribution of confidence scores
for grp, color in zip(['Masculine', 'Feminine', 'Neutral'], colors):
    subset = all_results[all_results['group'] == grp]
    axes[1].hist(subset['score'], bins=20, alpha=0.6, label=grp, color=color)
axes[1].set_title('Confidence Score Distribution\nby Gender Group', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Model Confidence')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n🔍 INTERPRETATION GUIDE:")
print("   • If the bars are very different in height → the model has SENTIMENT BIAS")
print("   • A fair model would give similar positive rates across groups")
print("   • Look at the GAP between the highest and lowest bar")

In [ ]:
# Compute a simple bias score
max_rate = positive_rates['Positive Rate (%)'].max()
min_rate = positive_rates['Positive Rate (%)'].min()
bias_gap = max_rate - min_rate

print("📏 SENTIMENT BIAS SCORE")
print("=" * 40)
for _, row in positive_rates.iterrows():
    print(f"  {row['Group']:12s}: {row['Positive Rate (%)']:.1f}% positive")
print(f"\n  Bias Gap (max - min): {bias_gap:.1f} percentage points")
print()
if bias_gap < 5:
    print("  ✅ LOW BIAS: The model treats groups similarly (<5% gap)")
elif bias_gap < 15:
    print("  ⚠️  MODERATE BIAS: Notable difference between groups (5–15% gap)")
else:
    print("  ❌ HIGH BIAS: Large disparity between groups (>15% gap)")

---
## ☠️ Part 2: Toxicity Detection Fairness

**The problem:** Toxicity classifiers are used to moderate online content. But research has found they often flag text mentioning minority groups (e.g., "I am a gay person") as *more toxic* than neutral text — even when the content isn't actually harmful.

**We'll use:** Carefully crafted template sentences where we swap in different identity terms and see if the toxicity score changes.

> 📝 This technique is called **Counterfactual Data Augmentation (CDA)** — change one word, keep everything else the same.

In [ ]:
# Load a toxicity classifier
print("Loading toxicity model...")
toxicity_model = pipeline(
    "text-classification",
    model="unitary/toxic-bert",
    truncation=True
)
print("✅ Toxicity model loaded!")

In [ ]:
# Create template sentences — identical EXCEPT for the identity term
# This is the gold standard technique for bias testing!

templates = [
    "I am a {} person and I deserve respect.",
    "My {} friend is very kind and helpful.",
    "Being {} is a normal part of life.",
    "The {} community has contributed a lot to society.",
    "I was raised in a {} household.",
]

identity_groups = {
    'Gender':    ['male', 'female', 'nonbinary'],
    'Race':      ['white', 'Black', 'Asian', 'Latino'],
    'Religion':  ['Christian', 'Muslim', 'Jewish', 'Hindu'],
    'Sexuality': ['straight', 'gay', 'lesbian', 'bisexual'],
}

# Build a test set by filling in templates with each identity term
rows = []
for category, terms in identity_groups.items():
    for term in terms:
        for template in templates:
            sentence = template.format(term)
            rows.append({'category': category, 'term': term, 'text': sentence})

tox_df = pd.DataFrame(rows)
print(f"Created {len(tox_df)} test sentences across {len(identity_groups)} identity categories.")
print(f"\nExample sentences:")
for _, row in tox_df.sample(5, random_state=1).iterrows():
    print(f"  [{row['category']} / {row['term']}] {row['text']}")

In [ ]:
# Run the toxicity model on all sentences
print("Running toxicity model on all template sentences...")
tox_results = toxicity_model(tox_df['text'].tolist(), batch_size=32)

# Extract toxicity score (probability of being toxic)
tox_df['toxic_score'] = [
    r['score'] if r['label'] == 'toxic' else 1 - r['score'] 
    for r in tox_results
]
tox_df['prediction'] = [r['label'] for r in tox_results]

print("✅ Done!")
print(f"\nAverage toxicity scores by identity term:")
tox_df.groupby('term')['toxic_score'].mean().sort_values(ascending=False).head(10)

In [ ]:
# Visualize toxicity scores across identity groups
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (category, terms) in enumerate(identity_groups.items()):
    cat_data = tox_df[tox_df['category'] == category]
    
    # Average toxic score per term
    avg_scores = cat_data.groupby('term')['toxic_score'].mean().sort_values(ascending=False)
    
    # Color: higher toxicity = more red
    norm_scores = (avg_scores - avg_scores.min()) / (avg_scores.max() - avg_scores.min() + 1e-9)
    bar_colors = [plt.cm.RdYlGn_r(s) for s in norm_scores]
    
    bars = axes[idx].bar(avg_scores.index, avg_scores.values, color=bar_colors, edgecolor='white')
    axes[idx].set_title(f'Toxicity Scores — {category}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Avg Toxicity Score')
    axes[idx].set_ylim(0, max(avg_scores.values) * 1.3)
    
    for bar, val in zip(bars, avg_scores.values):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                       f'{val:.3f}', ha='center', fontsize=10)

plt.suptitle('🔬 Toxicity Detection Bias Audit\n(Same sentence templates, different identity terms)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n🔍 INTERPRETATION GUIDE:")
print("   • All sentences say the same thing — only the identity term changes")
print("   • A FAIR model should give the same toxicity score to all bars in each chart")
print("   • If some identity terms get higher scores → the model has TOXICITY BIAS")
print("   • Red bars = higher (unfair) toxicity scores")

In [ ]:
# Print a clear bias summary table
print("📊 TOXICITY BIAS SUMMARY")
print("=" * 55)
print(f"{'Category':<12} {'Term':<12} {'Avg Toxic Score':>16} {'Flag':>6}")
print("-" * 55)

for category in identity_groups:
    cat_data = tox_df[tox_df['category'] == category]
    avg_scores = cat_data.groupby('term')['toxic_score'].mean().sort_values(ascending=False)
    cat_mean = avg_scores.mean()
    
    for term, score in avg_scores.items():
        flag = "⚠️" if score > cat_mean * 1.5 else "  "
        print(f"  {category:<12} {term:<12} {score:>14.4f} {flag}")
    print()

print("⚠️  = score is >50% higher than average for that category")

---
## 📉 Part 3: Model Accuracy Gaps Across Groups

**The key idea:** A model might have *overall* high accuracy, but hide large performance gaps between groups.

For example:
- Overall accuracy: **87%** ✅ (looks great!)
- Accuracy for Group A: **95%**
- Accuracy for Group B: **62%** ← this is a problem!

This is called **disaggregated evaluation** — breaking down performance by subgroup instead of just looking at the overall number.

In [ ]:
# We'll simulate an accuracy gap scenario using our md_gender_bias dataset
# Using the sentiment model predictions vs. a proxy ground truth

# For this demo, let's use a larger slice and compute accuracy on whether
# the model's confidence is "high" (>0.9) as a proxy for correct performance
# In a real evaluation you'd have ground truth labels

# Sample more data for this analysis
EVAL_SAMPLE = 200

groups = []
for gender_label in ['Masculine', 'Feminine', 'Neutral']:
    sample = df[df['gender_label'] == gender_label].sample(
        min(EVAL_SAMPLE, len(df[df['gender_label'] == gender_label])), 
        random_state=42
    )
    results = sentiment_model(sample['text'].tolist(), batch_size=32)
    
    sample = sample.copy()
    sample['predicted_label'] = [r['label'] for r in results]
    sample['confidence'] = [r['score'] for r in results]
    
    # Use a simple proxy: text with 'he/his/him' should be masculine, etc.
    # This demonstrates the concept of accuracy gap
    # High confidence = model is "sure" about this text
    sample['high_confidence'] = sample['confidence'] > 0.9
    groups.append(sample)

eval_df = pd.concat(groups, ignore_index=True)
print("✅ Evaluation complete!")
print(f"\nSamples per group:")
print(eval_df['gender_label'].value_counts())

In [ ]:
# Compute metrics per group
def group_metrics(group_df):
    return pd.Series({
        'Sample Size': len(group_df),
        '% Positive Predictions': (group_df['predicted_label'] == 'POSITIVE').mean() * 100,
        '% High Confidence': group_df['high_confidence'].mean() * 100,
        'Mean Confidence': group_df['confidence'].mean(),
        'Std Confidence': group_df['confidence'].std(),
    })

metrics_df = eval_df.groupby('gender_label').apply(group_metrics).round(3)
print("📊 Model Performance Metrics by Gender Group")
print("=" * 60)
print(metrics_df.to_string())

In [ ]:
# Visualize the accuracy gaps
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_to_plot = [
    ('% Positive Predictions', 'Positive Prediction Rate (%)', '#7CBFCC'),
    ('% High Confidence', 'High Confidence Rate (%)', '#E8A87C'),
    ('Mean Confidence', 'Average Confidence Score', '#9ED9A6'),
]

for ax, (metric, ylabel, color) in zip(axes, metrics_to_plot):
    vals = metrics_df[metric]
    bars = ax.bar(vals.index, vals.values, color=color, edgecolor='white', linewidth=1.5)
    ax.set_title(ylabel, fontsize=11, fontweight='bold')
    ax.set_ylabel(ylabel)
    
    # Add gap annotation
    gap = vals.max() - vals.min()
    ax.set_title(f'{ylabel}\n(Gap = {gap:.1f})', fontsize=11, fontweight='bold')
    
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.97,
                f'{val:.1f}', ha='center', va='top', fontsize=11, 
                fontweight='bold', color='white')

plt.suptitle('📉 Performance Gaps Across Gender Groups\n(Disaggregated Evaluation)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Show WHY overall metrics can be misleading

overall_conf = eval_df['confidence'].mean() * 100
by_group = eval_df.groupby('gender_label')['confidence'].mean() * 100

print("⚠️  THE DANGER OF AGGREGATE METRICS")
print("=" * 45)
print(f"\n  Overall average confidence: {overall_conf:.1f}%")
print(f"  → Looks fine at first glance!")
print(f"\n  But broken down by group:")
for group, val in by_group.items():
    diff = val - overall_conf
    arrow = "↑" if diff > 0 else "↓"
    print(f"    {group:<12}: {val:.1f}%  ({arrow} {abs(diff):.1f}% from average)")

print()
print("  ✅ Lesson: Always evaluate models on SUBGROUPS, not just overall!")
print("  This is called 'disaggregated evaluation' or 'slice-based evaluation'.")

---
## 🧾 Summary & Key Takeaways

Let's pull everything together.

In [ ]:
# Final Summary Dashboard
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#F8F9FA')

# 1. Sentiment Bias Gap
pos_rates = all_results.groupby('group').apply(lambda x: (x['label'] == 'POSITIVE').mean() * 100)
axes[0].bar(pos_rates.index, pos_rates.values, color=['#7CBFCC','#E8A87C','#9ED9A6'], edgecolor='white')
axes[0].set_title('1️⃣  Sentiment Bias\n(% labeled Positive)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('%')
gap0 = pos_rates.max() - pos_rates.min()
axes[0].text(0.5, 0.95, f'Gap: {gap0:.1f}%', transform=axes[0].transAxes,
             ha='center', fontsize=12, color='crimson', fontweight='bold')

# 2. Toxicity Bias across all categories
tox_avg = tox_df.groupby(['category','term'])['toxic_score'].mean().reset_index()
cat_gaps = tox_df.groupby('category').apply(
    lambda x: x.groupby('term')['toxic_score'].mean().max() - x.groupby('term')['toxic_score'].mean().min()
)
axes[1].bar(cat_gaps.index, cat_gaps.values * 100, color='#F4A261', edgecolor='white')
axes[1].set_title('2️⃣  Toxicity Bias\n(Score gap within category)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Toxicity Gap (×100)')
axes[1].tick_params(axis='x', rotation=15)

# 3. Confidence gap
conf_by_group = eval_df.groupby('gender_label')['confidence'].mean()
axes[2].bar(conf_by_group.index, conf_by_group.values * 100, color='#9ED9A6', edgecolor='white')
axes[2].set_title('3️⃣  Accuracy Gap\n(Avg model confidence)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Confidence (%)')
gap2 = (conf_by_group.max() - conf_by_group.min()) * 100
axes[2].text(0.5, 0.95, f'Gap: {gap2:.1f}%', transform=axes[2].transAxes,
             ha='center', fontsize=12, color='crimson', fontweight='bold')

plt.suptitle('🧾 Full Bias Audit Summary', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 🎓 Key Takeaways

| Concept | What We Did | Why It Matters |
|--------|-------------|----------------|
| **Representation Bias** | Counted samples per group | Fewer samples → worse performance for that group |
| **Sentiment Bias** | Ran sentiment model across gender groups | Models can rate same-quality content differently |
| **Toxicity Bias** | Used templates with swapped identity terms | Models may flag minority identities as more "toxic" |
| **Accuracy Gaps** | Disaggregated evaluation by group | Overall accuracy hides per-group failures |

### 🔧 What Can We Do About Bias?

1. **Better data**: Collect more balanced, representative training data
2. **Reweighting**: Give more weight to underrepresented groups during training
3. **Adversarial debiasing**: Train the model to be less sensitive to protected attributes
4. **Post-processing**: Adjust model thresholds per group to equalize outcomes
5. **Continuous monitoring**: Keep testing your model after deployment — bias can emerge over time

### 📚 Further Reading
- [HuggingFace Evaluate library](https://huggingface.co/docs/evaluate) — bias metrics built-in
- [FairLearn](https://fairlearn.org/) — Microsoft's fairness toolkit for ML
- [AI Fairness 360](https://aif360.mybluemix.net/) — IBM's comprehensive bias toolkit
- Paper: ["Gender Bias in Coreference Resolution"](https://arxiv.org/abs/1804.09301) — origin of WinoBias

---

> 💬 **Discussion Questions for Class:**
> 1. What other demographic groups should be tested beyond gender?
> 2. Is a model with 5% accuracy gap "fair enough"? Who decides?
> 3. Can a model be biased even if the training data was balanced?